In [1]:
# visualization strategies
# cell miroscopy motility data
#correlation calculator for cell tracking data
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import pdb
from scipy import integrate
import matplotlib as mpl
from scipy import interpolate
import time

from ABM_package import *
from PDE_FIND3 import * 
from model_selection_IP3 import *
import time, glob
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png', 'pdf')


import os
from correlation_package import *

#increase font size throughout
font = {'size'   : 25}
plt.rc('font', **font)
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")  # Use GPU
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")   # Fallback to CPU
    print("GPU not available, using CPU")

Using GPU: NVIDIA GeForce RTX 5090 Laptop GPU


In [2]:
folder= "../data/processed_coculture/"

c_dfs={}

import os
import pickle
import pandas as pd

folder = "../data/processed_coculture/"
c_dfs = {}

for fname in os.listdir(folder):
    if fname.endswith(".pkl"):
        full_path = os.path.join(folder, fname)
        
        with open(full_path, "rb") as f:
            obj = pickle.load(f)

        df = pd.DataFrame(obj)

        # Get base name without .pkl extension
        base_name = os.path.splitext(fname)[0]

        # Split by each unique experiment value
        unique_experiments = df['experiment'].unique()
        
        for exp in unique_experiments:
            df_exp = df[df['experiment'] == exp].reset_index(drop=True)
            key = f"{base_name}_{exp}"
            c_dfs[key] = df_exp
            print(f"Loaded {key} → {df_exp.shape} rows")

/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)
/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, us

Loaded Fibronectin_0_625_-40-F5-F05 → (35582, 18) rows
Loaded Fibronectin_0_625_-41-F6-F06 → (34416, 18) rows
Loaded Fibronectin_0_625_-42-F7-F07 → (47539, 18) rows
Loaded Plastic_0_313_-46-G2-G02 → (55613, 18) rows
Loaded Plastic_0_313_-47-G3-G03 → (45646, 18) rows
Loaded Plastic_0_313_-48-G4-G04 → (27519, 18) rows
Loaded Fibronectin_10_-04-B5-B05 → (551600, 18) rows
Loaded Fibronectin_10_-05-B6-B06 → (449409, 18) rows
Loaded Fibronectin_10_-06-B7-B07 → (1004315, 18) rows
Loaded PDK_0_313_-52-G8-G08 → (37154, 18) rows
Loaded PDK_0_313_-53-G9-G09 → (47878, 18) rows
Loaded PDK_0_313_54-G10-G10 → (14461, 18) rows


/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)
/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, us

Loaded Plastic_10_corrected_density_-01-B2-B02 → (428351, 18) rows
Loaded Plastic_10_corrected_density_-02-B3-B03 → (334287, 18) rows
Loaded Plastic_10_corrected_density_-03-B4-B04 → (344026, 18) rows
Loaded Plastic_0_625_-37-F2-F02 → (55452, 18) rows
Loaded Plastic_0_625_-38-F3-F03 → (29832, 18) rows
Loaded Plastic_0_625_-39-F4-F04 → (34926, 18) rows
Loaded Plastic_1_25_-28-E2-E02 → (79827, 18) rows
Loaded Plastic_1_25_-29-E3-E03 → (27544, 18) rows


/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)
/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, us

...

Loaded Plastic_2_5_-19-D2-D02 → (140887, 18) rows
Loaded Plastic_2_5_-20-D3-D03 → (157831, 18) rows
Loaded Plastic_2_5_-21-D4-D04 → (126825, 18) rows
Loaded PDK_0_625_no44_-43-F8-F08 → (45493, 18) rows
Loaded PDK_0_625_no44_45-F10-F10 → (26262, 18) rows
Loaded Plastic_5_corrected_density_-10-C2-C02 → (212527, 18) rows
Loaded Plastic_5_corrected_density_-11-C3-C03 → (233219, 18) rows
Loaded Plastic_5_corrected_density_-12-C4-C04 → (676876, 18) rows
Loaded Fibronectin_0_313_-49-G5-G05 → (32453, 18) rows
Loaded Fibronectin_0_313_-50-G6-G06 → (38600, 18) rows
Loaded Fibronectin_0_313_-51-G7-G07 → (50510, 18) rows


/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)
/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, us

Loaded PDK_1_25_-34-E8-E08 → (64106, 18) rows
Loaded PDK_1_25_-35-E9-E09 → (75973, 18) rows
Loaded PDK_1_25_36-E10-E10 → (40626, 18) rows
Loaded PDK_2_5_-25-D8-D08 → (223555, 18) rows
Loaded PDK_2_5_-26-D9-D09 → (212992, 18) rows
Loaded PDK_2_5_27-D10-D10 → (68953, 18) rows


/tmp/ipykernel_24848/3084661762.py:17: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)


Loaded PDK_5_corrected_density_-16-C8-C08 → (559613, 18) rows
Loaded PDK_5_corrected_density_-17-C9-C09 → (434147, 18) rows
Loaded PDK_5_corrected_density_18-C10-C10 → (134149, 18) rows


In [3]:


folder = "../data/processed_monoculture/"

m_dfs = {}  # store each df with its filename

for fname in os.listdir(folder):
    if fname.endswith(".pkl"):
        full_path = os.path.join(folder, fname)
        
        with open(full_path, "rb") as f:
            obj = pickle.load(f)
        
        # Convert to DataFrame
        df = pd.DataFrame(obj)
        m_dfs[fname] = df

        print(f"Loaded {fname} → {df.shape} rows")

Loaded d16_linked.pkl → (356278, 16) rows
Loaded Wildtype_linked.pkl → (495266, 16) rows
Loaded p95_linked.pkl → (452775, 16) rows


/tmp/ipykernel_24848/1551367962.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  obj = pickle.load(f)
/tmp/ipykernel_24848/1551367962.py:10: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, us

In [4]:
mapping = {
    "Y": 0,
    "M": 1,
    "T": 2
}

for fname in m_dfs.keys():
    df = m_dfs[fname]
    df["phenotype"] = df["color"].map(mapping)
    m_dfs[fname] = df

for fname in c_dfs.keys():
    df = c_dfs[fname]
    df["phenotype"] = df["color"].map(mapping)
    c_dfs[fname] = df

In [5]:
df_current= c_dfs['Fibronectin_0_625_-40-F5-F05']

df_current.head()

,frame,y,x,particle,area,equiv_diameter,mean_c0,mean_c1,mean_c2,std_c0,std_c1,std_c2,color,experiment,initial_density,coating,x_microns,y_microns,phenotype
0,0,3.985401,57.416058,0,137.0,13.207340,51.583942,47.810219,44.620438,7.379833,3.168119,0.960014,Y,-40-F5-F05,0.625,Fibronectin,79.004496,5.483912,0
1,0,869.683250,266.263682,1,603.0,27.708545,52.711443,47.862355,47.363184,6.560573,0.763573,1.746573,M,-40-F5-F05,0.625,Fibronectin,366.378826,1196.684153,1
2,0,854.154286,227.365714,2,175.0,14.927053,47.405714,50.388571,48.697143,1.621629,3.860109,3.473120,T,-40-F5-F05,0.625,Fibronectin,312.855223,1175.316297,2
3,0,846.248344,217.867550,3,302.0,19.609139,65.582781,49.996689,51.874172,22.662469,3.372560,7.484027,M,-40-F5-F05,0.625,Fibronectin,299.785748,1164.437722,1
4,0,797.541401,974.452229,4,157.0,14.138550,45.745223,49.872611,48.108280,1.266443,3.215659,2.914011,T,-40-F5-F05,0.625,Fibronectin,1340.846268,1097.416968,2
